Now there are some problem that we need to resolve, in csv the columns contains date data type have inconsistent data like year is yyyy and some time yy, some more date issues you know. Same problem with date time datatype columns year is yyyy, sometimes yy, and in time some time am,pm is present and some time not, some time sec is present sometime not. i hope you understand otherwise you can ask me. More issue is that if we clean columns contains date then the date time columns also converted into date datatype that i dont wanted, I want date as date and date time as date time datatype. I tell you the date time column name same as present in csv file = Acquisition Date/Time and Time of Sale (for time datatype). please give me an updated code , you can make changes in my code but highlight the changes you make also tell me logic as comment there. If you understand then thumbs up and if you have any confusion then ask me then code, you also find anomalies in datatypes for all columns by codding

### **Updated Code**
1. This code chages the encoding in latin1, if not possible in utf-8.
2. This also correct the date and datetime columns.
3. This code TRUNCATE the data and uplaods new data if want to add new data then use APPEND.
4. If APPEND is used this code doesnt fillter out duplicate data before sending it to BigQuery.

In [ ]:
import os
import pandas as pd
from google.cloud import bigquery
from datetime import datetime
from collections import Counter
from tqdm import tqdm  # ✅ for progress bar


In [ ]:

# ✅ Step 1: Set credentials
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = '/Users/piyushpanthi/Desktop/Data Analyst Projects/variant-data-analytics-bb72356ef903.json'

# ✅ Step 2: Set BigQuery details
project_id = "variant-data-analytics"
dataset_id = "Sticky_Data"
table_name = "Sticky_PD"
table_id = f"{project_id}.{dataset_id}.{table_name}"

# ✅ Step 3: Define your folder path
folder_path =  '/Users/piyushpanthi/Desktop/Verve Advisory/Sticky.io Data/Sticky.io PD'


In [ ]:

# ✅ Step 4: Read and combine CSVs
all_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
df_list = []

for file in all_files:
    path = os.path.join(folder_path, file)
    try:
        # 🔄 Try UTF-8 first, then fall back to latin1
        try:
            df = pd.read_csv(path, encoding="utf-8", on_bad_lines="skip")
        except UnicodeDecodeError:
            print(f"⚠️ Retrying {file} with latin1 encoding...")
            df = pd.read_csv(path, encoding="latin1", on_bad_lines="skip")

        # ✅ Check if any column names look like data (e.g., emails or numbers)
        bad_header = any('@' in str(col) or str(col).strip().isdigit() for col in df.columns)
        if bad_header:
            print(f"⚠️ Detected invalid header in {file}. Skipping first row and reloading.")
            try:
                df = pd.read_csv(path, encoding="utf-8", skiprows=1, on_bad_lines="skip")
            except UnicodeDecodeError:
                df = pd.read_csv(path, encoding="latin1", skiprows=1, on_bad_lines="skip")

        df.columns = [str(col).strip() for col in df.columns]
        df_list.append(df)

    except Exception as e:
        print(f"⚠️ Skipped {file} due to error: {e}")

combined_df = pd.concat(df_list, ignore_index=True)

# ✅ Check for invalid columns (emails or numbers as headers)
invalid_cols = [col for col in combined_df.columns if '@' in col or col.strip().isdigit()]
if invalid_cols:
    print("⚠️ Invalid columns detected that look like data values (e.g., emails):")
    print(invalid_cols)
    raise ValueError("❌ CSV headers are likely incorrect. Please inspect the files.")


In [ ]:

# ✅ Step 5: Clean column names
combined_df.columns = [col.strip().replace(' ', '_').replace('-', '_').replace('/', '_') for col in combined_df.columns]


In [ ]:

# ✅ Step 6: Deduplicate column names
col_counter = Counter()
new_cols = []
for col in combined_df.columns:
    if col_counter[col] == 0:
        new_cols.append(col)
    else:
        new_cols.append(f"{col}_{col_counter[col]}")
    col_counter[col] += 1
combined_df.columns = new_cols


# 🔴 Custom parsers
def parse_mixed_datetime(val):
    import dateutil.parser
    if pd.isna(val):
        return pd.NaT
    try:
        dt = dateutil.parser.parse(str(val), fuzzy=True)
        if dt.year < 1000:
            dt = dt.replace(year=dt.year + 2000)
        return dt
    except Exception:
        return pd.NaT

def parse_mixed_date(val):
    if pd.isna(val):
        return pd.NaT
    try:
        dt = pd.to_datetime(val, errors='coerce', infer_datetime_format=True)
        if dt is pd.NaT or pd.isna(dt):
            return pd.NaT
        if dt.year < 1000:
            dt = dt.replace(year=dt.year + 2000)
        return dt.date()
    except Exception:
        return None

def parse_mixed_time(val):
    import dateutil.parser
    if pd.isna(val):
        return None
    try:
        dt = dateutil.parser.parse(str(val), fuzzy=True)
        return dt.time()
    except Exception:
        return None


In [ ]:


# ✅ Step 7: Define schema + cast dtypes
schema = []
print("\n🔍 Inferring schema and converting data types...")

for col in tqdm(combined_df.columns, desc="🔄 Processing columns"):
    field_type = "STRING"  # Default
    col_lower = col.lower()

    float_fields = {
        "Ship_Price", "Non_Taxable_Total", "Taxable_Total", "Sub_Total",
        "Sales_Tax_Percent", "Sales_Tax_Factor", "Order_Total", "Gateway_Processing_Percent",
        "Gateway_Reserve_Percent", "Gateway_Transaction_Fee", "Gateway_Chargeback_Fee",
        "Total_Installments", "Void_Amount", "Refund_Amount", "Rebill_Discount",
        "Product_Price", "Quantity", "Declared_Value", "Decline_Salvage_Discount",
        "Decline_Salvage_Discount_%"
    }

    # Explicit datetime fields ( CHANGED: added Time_of_Sale here)
    datetime_fields = {"Acquisition_Date_Time", "Time_of_Sale"}

    time_fields = set()  #  CHANGED: no Time_of_Sale here anymore

    date_fields = {
        "Date_of_Sale", "Shipped_Date", "Fraud_Date", "Chargeback_Date",
        "RMA_Date", "Recurring_Date", "Retry_Date", "Hold_Date", "Void_Date",
        "Refund_Date", "Confirmation_Date"
    }

    int_fields = {"Billing_Cycle"}

    string_fields = {"Order_Id", "Created_By", "Bill_First", "Bill_Last", "Bill_Address1", "Bill_Address2",
                    "Bill_City", "Bill_State", "Bill_Zip", "Bill_Country", "Bill_Phone", "Bill_Email",
                    "Ship_First", "Ship_Last", "Ship_Address1", "Ship_Address2", "Ship_City", "Ship_State",
                    "Ship_Zip", "Ship_Country", "Ship_Phone", "Ship_Email", "Ship_Method_Name",
                    "Ship_Method_Description", "Ship_Group_Name", "Ship_Group_Code", "Weight",
                    "Delivery_Confirmation", "Signature_Confirmation", "Tracking_Number", "Payment",
                    "Campaign_Id", "Customer_Number", "Prospect_Number", "CNPJ_CPF_ID_Document_ID",
                    "Credit_Card_Number", "Credit_Card_Expiration", "Prepaid_Match", "Gateway_Id",
                    "Gateway_Alias", "Gateway_Descriptor", "Gateway_Customer_Service_Number",
                    "Processor_Id", "IP_Address", "IP_Address_Lookup", "Order_Status", "Decline_Reason",
                    "Is_Cascaded", "Original_Gateway_Id", "Original_Decline_Reason", "Is_Fraud",
                    "Is_Chargeback", "Chargeback_By", "Is_RMA", "RMA_Number", "RMA_Reason",
                    "RMA_Created_By", "Return", "Return_Reason", "Is_Recurring", "Transaction_Number",
                    "Auth_Number", "Retrying", "Retries_Left", "Retry_Attempt", "Is_Void", "Voided_By",
                    "Is_Refund", "Refunded_By", "Refund_Note", "AFID", "SID", "AFFID", "C1", "C2", "C3",
                    "BID", "AID", "OPT", "Parent_Order_Id", "Product_Id", "Product_Name",
                    "Product_Attributes", "Is_Product_Shippable", "Product_Sku_#", "Product_Category",
                    "Description", "Confirmation", "Next_Recurring_Product", "Next_Recurring_Product_Id",
                    "Blacklisted", "Ancestor_Order_Id", "Test", "Is_Cancel", "Hold_Type", "Currency",
                    "Offer_Id", "Billing_Model_Id", "Notes", "Promo_Code"}

    try:
        if col in float_fields:
            field_type = "FLOAT"
            combined_df[col] = pd.to_numeric(combined_df[col], errors="coerce")

        elif col in int_fields:
            field_type = "INT64"
            combined_df[col] = pd.to_numeric(combined_df[col], errors="coerce").astype('Int64')

        elif col in date_fields:
            field_type = "DATE"
            combined_df[col] = combined_df[col].apply(parse_mixed_date)

        elif col in datetime_fields:
            field_type = "DATETIME"
            combined_df[col] = combined_df[col].apply(parse_mixed_datetime)

        elif col in time_fields:
            field_type = "TIME"
            combined_df[col] = combined_df[col].apply(parse_mixed_time)

        elif col in string_fields:
            field_type = "STRING"
            combined_df[col] = combined_df[col].astype(str)

        else:
            if "date" in col_lower:
                field_type = "DATE"
                combined_df[col] = combined_df[col].apply(parse_mixed_date)
            elif "time" in col_lower or "timestamp" in col_lower:
                field_type = "DATETIME"
                combined_df[col] = combined_df[col].apply(parse_mixed_datetime)
            else:
                numeric_col = pd.to_numeric(combined_df[col], errors='coerce')
                if not numeric_col.dropna().empty:
                    if numeric_col.dropna().apply(float.is_integer).all():
                        combined_df[col] = numeric_col.astype('Int64')
                        field_type = "INT64"
                    else:
                        combined_df[col] = numeric_col
                        field_type = "FLOAT"
                else:
                    combined_df[col] = combined_df[col].astype(str)
                    field_type = "STRING"

    except Exception as e:
        print(f"⚠️ Could not process column {col}, defaulting to STRING. Error: {e}")
        combined_df[col] = combined_df[col].astype(str)
        field_type = "STRING"

    schema.append(bigquery.SchemaField(col, field_type))


In [ ]:

# ✅ Step 8: Replace NaNs with None
combined_df = combined_df.where(pd.notnull(combined_df), None)

# ✅ Step 8.5: Check type consistency
print("\n🧪 Inconsistent data types detected (non-null type counts per column):")
for col in combined_df.columns:
    type_counts = combined_df[col].apply(lambda x: type(x).__name__).value_counts()
    if len(type_counts) > 1:
        print(f"⚠️ {col} has mixed types:")
        print(type_counts)


In [ ]:

# ✅ Step 9: Upload to BigQuery
print("\n🚀 Uploading data to BigQuery...")
client = bigquery.Client()
job_config = bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_APPEND")

job = client.load_table_from_dataframe(combined_df, table_id, job_config=job_config)
job.result()

print(f"\n✅ Successfully uploaded {len(combined_df)} rows to {table_id}")